<div style="max-width: 80ch; white-space: normal;">

# Лекция 1: Введение в машинное обучение

## 1. Базовые понятия: Медицинская рекомендательная система

Главный пример лекции — **умный помощник врача**, который анализирует карту пациента и подбирает наиболее похожие диагнозы из исторического справочника (100 000 прошлых осмотров).

* **Объект ($X$)**: Пара «Данные осмотра Пациента — Текстовый диагноз из справочника».
* **Ответ ($Y$)**: Выбрал ли врач в итоге этот (или семантически идентичный) диагноз? (1 — да, 0 — нет).
* **Признаки ($x$):** 
  * *Признаки пациента (29 шт.):* 8 числовых, 20 choice-параметров, 1 OHE (тоны сердца).
  * *Признаки диагноза:* Только сырой текст и аббревиатуры (без кодов МКБ-10).
  * **Парные признаки (NLP):** Насколько текст жалоб и симптомов *семантически похож* на текст диагноза из справочника? (Алгоритм понимает, что "ХСН" и "Серд. недост." — это близко).

</div>


In [2]:
import pandas as pd
import numpy as np

# Теперь диагнозы - это грязный текст с сокращениями, а точного совпадения может не быть.
# Поэтому появляется метрика "Семантической близости" (от 0 до 1)

data = {
    'patient_id': [1, 1, 2, 2],
    
    # --- Признаки пациента ---
    'age': [65, 65, 40, 40],               
    'systolic_bp': [150, 150, 120, 120],
    'doctor_notes': ['одышка, слаб., отеки', 'одышка, слаб., отеки', 'здоров, жалоб нет', 'здоров, жалоб нет'],
    
    # --- Признаки диагноза (сырой текст из справочника) ---
    'catalog_diag_text': ['ХСН 2 ст.', 'Гастрит обостр.', 'Норма', 'ОРВИ'],
    
    # --- ПАРНЫЙ ПРИЗНАК: Семантическая близость (Text Similarity Score) ---
    # Насколько записи врача по смыслу похожи на текст из справочника
    'semantic_similarity': [0.89, 0.12, 0.95, 0.30],
    
    # Целевая переменная (Y) - это то, что мы предсказываем
    'is_relevant': [1, 0, 1, 0]
}

df = pd.DataFrame(data)
display(df)

,patient_id,age,systolic_bp,doctor_notes,catalog_diag_text,semantic_similarity,is_relevant
0,1,65,150,"одышка, слаб., отеки",ХСН 2 ст.,0.89,1
1,1,65,150,"одышка, слаб., отеки",Гастрит обостр.,0.12,0
2,2,40,120,"здоров, жалоб нет",Норма,0.95,1
3,2,40,120,"здоров, жалоб нет",ОРВИ,0.30,0


## 2. Типы признаков

Компьютер не понимает букв, поэтому текст нужно превращать в числа:

* **Числовые**: Наши 8 параметров (Возраст, давление).
* **Категориальные (choice)**: 20 параметров. Переводятся в OHE.
* **Текстовые признаки**: Текст жалоб пациента, анамнез или записи терапевта.
* **Бинарные признаки**: Пол (мужской/женский), наличие кардиостимулятора (да/нет).
* **Инженерные фичи**: Сколько раз этот пациент обращался за последний год, среднее давление за последние 3 визита.
* **Составные признаки**: Совпадает ли возрастная группа пациента с типичным возрастом для этого диагноза.
* **Векторные представления (Embeddings)**: это инструмент перевода (мост) из сложного мира неструктурированных данных (текстов, картинок, звуков) в мир чисел. Может быть признаком для нейросети.

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Создадим демонстрационный датасет для одного пациента и двух кандидатов-диагнозов
data = {
    # 1. Числовые признаки
    "age": [58, 58],
    "systolic_bp": [140, 140],
    
    # 2. Категориальные признаки (сырые)
    "smoking_status": ["never", "never"],
    
    # 4. Бинарные признаки (0 или 1)
    "is_male": [1, 1],
    "has_pacemaker": [0, 0],
    
    # 5. Инженерные фичи (созданные на основе истории)
    "visits_last_year": [4, 4],
    "avg_bp_last_3_visits": [135.0, 135.0],
    
    # Кандидаты из справочника диагнозов
    "candidate_diagnosis": [
        "Хроническая сердечная недостаточность II ст.", 
        "Острый гастрит"
    ]
}

df = pd.DataFrame(data)

# --- Обработка признаков ---

# 3. Текстовые признаки и 7. Векторные представления (Embeddings)
# В реальности мы бы использовали SentenceTransformers, здесь имитируем эмбеддинги
complaint_text = "Пациент жалуется на сильную одышку при нагрузке и отеки ног."
print(f"Текстовый признак (жалобы): {complaint_text}\n")

# Имитируем векторизацию (эмбеддинги текста жалобы и текстов диагнозов)
np.random.seed(42)
complaint_embedding = np.random.rand(1, 10) # В реальности размерность 384 или 768
candidate_embeddings = np.random.rand(2, 10)

# 6. Составные признаки (Interaction Features): считаем семантическую близость через косинусное расстояние
df["semantic_similarity"] = cosine_similarity(complaint_embedding, candidate_embeddings)[0]

# Категориальные признаки -> One-Hot Encoding (OHE)
encoder = OneHotEncoder(sparse_output=False, drop="first")
ohe_encoded = encoder.fit_transform(df[["smoking_status"]])
ohe_df = pd.DataFrame(ohe_encoded, columns=encoder.get_feature_names_out(["smoking_status"]))

# Собираем итоговую таблицу признаков для модели ранжирования
final_features = pd.concat([df.drop(columns=["smoking_status"]), ohe_df], axis=1)

print("Итоговая матрица признаков для модели:")
print(final_features[["age", "is_male", "visits_last_year", "semantic_similarity", "candidate_diagnosis"]])

Текстовый признак (жалобы): Пациент жалуется на сильную одышку при нагрузке и отеки ног.

Итоговая матрица признаков для модели:
   age  is_male  visits_last_year  semantic_similarity  \
0   58        1                 4             0.907777   
1   58        1                 4             0.667675   

                            candidate_diagnosis  
0  Хроническая сердечная недостаточность II ст.  
1                                Острый гастрит  


## 3. Архитектура решения (Двухэтапная система)

### Этап 1: Векторный поиск (Retrieval)
Поиск по 100 000 строк справочника через `LIKE '%...%'` не сработает из-за опечаток и сокращений.
* **Как работает:** Мы переводим симптомы текущего пациента в математический вектор и ищем в справочнике диагнозы с ближайшими векторами (считаем косинусное расстояние). Отбираем 30 «похожих» кандидатов.

### Этап 2: ML-Ранжирование (Reranking)
Обучаем модель оценивать отобранных кандидатов.
* Модель смотрит на парный признак `semantic_similarity`, добавляет к нему числовые анализы пациента (29 параметров) и решает, какой из похожих диагнозов наиболее вероятен в данном конкретном случае.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Обучаем ранжировщик (Этап 2)
# Модель учится балансировать между текстовой похожестью и физическими симптомами
X = df[['age', 'systolic_bp', 'semantic_similarity']]
Y = df['is_relevant']

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, Y)

# Вероятности для кандидатов
probabilities = model.predict_proba(X)[:, 1]

print("ML-оценка релевантности диагнозов:")
for i, prob in enumerate(probabilities):
    print(f"Текст врача: '{df['doctor_notes'].iloc[i]}' | Кандидат: '{df['catalog_diag_text'].iloc[i]}' -> Уверенность ML: {prob*100:.1f}%")

## 4. Интересные примеры и отступления из лекции

### Машинный перевод (почему не работают правила)
Если пытаться связать диагнозы правилами, мы сломаемся на первой же опечатке. 
> **Пример:** Классические пиратские переводы игр от «Фаргуса», где «Cool story» переводилось как «Прохладная история». В медицине то же самое: нельзя жестко запрограммировать, что «ХСН» = «Сердечная недостаточность», потому что завтра врач напишет «Срдеч. недост.». ML и векторы ищут смысл, а не точное совпадение букв.

### Как нельзя называть машинное обучение
Лектор просит **не называть машинное обучение словом «машинка»**. Это звучит пренебрежительно (сравнимо с тем, как матанализ называют «матаном» — это нормально, а вот греческий язык «гречкой» — уже странно).

   ### Почему стандартный подход начинает ломаться?
   
   Если мы жестко привязаны только к тем 100 диагнозам, что есть в справочнике, а врач написал в карте что-то уникальное (или использовал редкую формулировку), то:
   В нашей исторической базе таких «правильных пар» ($Y = 1$) может вообще не найтись.Модель будет пытаться натянуть сову на глобус, выбирая из 100 бедных вариантов то, что написал врач своими словами, и обучение «поплывет».
   
#### Вариант А: Нормализация и кластеризация (Чистим «историю»)
Прежде чем обучать модель, текстовые диагнозы терапевтов прогоняют через алгоритмы очистки (или LLM / эмбеддинги):

Все вариации вроде «ХСН 2ст», «серд. недост. II ст.», «ХСН ст. 2» с помощью векторного поиска схлопывают в один кластер и привязывают к базовому каноничному диагнозу из справочника.

Таким образом, мы расширяем справочник или автоматически «переводим» творчество терапевтов на язык справочника.

#### Вариант Б: Справочник становится динамическим
Справочник не высечен в камне. Если терапевты постоянно пишут новые адекватные формулировки:

Базовый справочник дополняется самыми частыми вариантами из практики (например, отсекаются редкие опечатки, а устойчивые сокращения врачей добавляются как официальные синонимы).

#### Вариант В: Генеративный подход (LLM / Generative AI)
В современных системах (вместо классического подбора из фиксированного списка) часто делают гибрид:

Модель с помощью векторного поиска находит в истории похожие случаи.

Большая языковая модель (LLM) смотрит на 29 параметров пациента, на подсказки из истории и сама формирует текст диагноза в стиле того самого терапевта, опираясь на лучшие практики из 300к записей.